# RAGAS Evaluation with Cost Analysis

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

assert os.environ.get("FIREWORKS_API_KEY"), "FIREWORKS_API_KEY missing from .env"
assert os.environ.get("GROQ_API_KEY"), "GROQ_API_KEY missing from .env"
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY missing from .env"
assert os.environ.get("LANGSMITH_API_KEY"), "LANGSMITH_API_KEY missing from .env"
assert os.environ.get("LANGSMITH_PROJECT"), "LANGSMITH_PROJECT missing from .env"
print(f"Tracing to LangSmith project: {os.environ['LANGSMITH_PROJECT']}")

Tracing to LangSmith project: Session10-LLM-Servers


In [2]:
# ragas 0.4.3 unconditionally imports langchain_community.chat_models.vertexai, a module
# that no longer ships in langchain-community>=0.4 (dropped in favor of the standalone
# langchain-google-vertexai package). We never use Vertex AI, so stub the module in
# sys.modules before importing ragas rather than pinning an older, incompatible langchain-community.
import sys
import types

if "langchain_community.chat_models.vertexai" not in sys.modules:
    _vertex_stub = types.ModuleType("langchain_community.chat_models.vertexai")

    class ChatVertexAI:
        pass

    _vertex_stub.ChatVertexAI = ChatVertexAI
    sys.modules["langchain_community.chat_models.vertexai"] = _vertex_stub

import asyncio
import pandas as pd
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings import OpenAIEmbeddings as RagasOpenAIEmbeddings
from ragas.metrics.collections import (
    AnswerRelevancy,
    ContextPrecisionWithReference,
    ContextRecall,
    Faithfulness,
)

## 1. Evaluation question set

In [3]:
# Questions + references drawn directly from data/cat-health-guide.pdf (2021 AAHA/AAFP
# Feline Life Stage Guidelines), so `reference` is grounded truth, not a guess -- this is
# what lets us score context_recall/context_precision.
eval_examples = [
    {
        "question": "What are the four life stages defined in the 2021 guidelines, and what age range does each cover?",
        "reference": (
            "Kitten (birth up to 1 year), young adult (1 through 6 years), mature adult "
            "(7 to 10 years), and senior (over 10 years), plus a fifth end-of-life stage "
            "that can occur at any age."
        ),
    },
    {
        "question": "How many life stages did the previous guidelines use, and how does the 2021 update change that?",
        "reference": (
            "The previous guidelines used six life stages. The 2021 update simplifies this "
            "to four distinct age-related stages (kitten, young adult, mature adult, senior) "
            "plus an end-of-life stage, for five stages overall."
        ),
    },
    {
        "question": "How often should senior cats be seen by a veterinarian?",
        "reference": "Senior cats should be seen at least every 6 months, and more frequently when indicated.",
    },
    {
        "question": "Which vaccines are considered core for cats?",
        "reference": (
            "Core vaccines are rabies virus, feline herpesvirus type 1 (FHV-1), feline "
            "calicivirus (FCV), and feline panleukopenia virus (FPV). Feline leukemia virus "
            "(FeLV) vaccination is also considered core for kittens."
        ),
    },
    {
        "question": "When should the FeLV vaccine be boosted after the kitten series?",
        "reference": (
            "Revaccinate for FeLV 12 months after the last dose in the kitten series, then "
            "annually for cats with high risk of regular exposure."
        ),
    },
    {
        "question": "What does the guide recommend for parasite control in cats?",
        "reference": (
            "Use broad-spectrum products effective against heartworms, intestinal parasites, "
            "and fleas; perform routine deworming; and check fecal samples regularly, since "
            "heartworm is harder to diagnose in cats and negative testing does not rule out infection."
        ),
    },
    {
        "question": "What is recommended regarding a cat's dental care starting in kittenhood?",
        "reference": (
            "Lifelong proactive dental care should be discussed starting at kitten wellness "
            "appointments -- checking for malocclusion or dental eruption problems early, "
            "encouraging tooth-brushing or VOHC-approved dental diets, and periodic "
            "professional dental prophylaxis with full oral radiography as the cat matures."
        ),
    },
    {
        "question": "What history and exam areas are most important during a kitten wellness visit?",
        "reference": (
            "Vaccination and parasite control history, health status of littermates and "
            "parents, nutritional status, and behavior -- including socialization and any "
            "unwanted behaviors -- are key discussion points at the kitten stage."
        ),
    },
]
print(f"{len(eval_examples)} evaluation questions loaded")

8 evaluation questions loaded


## 2. Run all three RAG pipelines

In [4]:
from app.rag import get_fireworks_rag_graph, get_groq_rag_graph, get_openai_rag_graph


def run_pipeline(graph, provider: str) -> list[dict]:
    rows = []
    for example in eval_examples:
        result = graph.invoke(
            {"question": example["question"]},
            config={"tags": [provider], "run_name": f"rag_eval_{provider}"},
        )
        rows.append(
            {
                "user_input": example["question"],
                "response": result["response"],
                "retrieved_contexts": [doc.page_content for doc in result["context"]],
                "reference": example["reference"],
            }
        )
    return rows

In [5]:
openai_rows = run_pipeline(get_openai_rag_graph(), "openai")
print("OpenAI sample response:", openai_rows[0]["response"][:200])

OpenAI sample response: The four life stages defined in the 2021 AAHA/AAFP Feline Life Stage Guidelines and their age ranges are:

1. Kitten: birth up to 1 year  
2. Young adult: 1 to 6 years  
3. Mature adult: 7 to 10 years


In [6]:
groq_rows = run_pipeline(get_groq_rag_graph(), "groq")
print("Groq sample response:", groq_rows[0]["response"][:200])

Groq sample response: The 2021 AAHA/AAFP Feline Life Stage Guidelines define four distinct life stages:

| Life stage | Age range |
|------------|-----------|
| **Kitten** | Birth to 1 year |
| **Young adult** | 1 to 6 yea


In [ ]:
fireworks_rows = run_pipeline(get_fireworks_rag_graph(), "fireworks")
print("Fireworks sample response:", fireworks_rows[0]["response"][:200])

## 3. Score with RAGAS

In [11]:
# Judge is fixed to OpenAI for both pipelines so neither is graded by its own provider.
# ragas.metrics.collections is instructor/async-based, so it needs an AsyncOpenAI client.
judge_client = AsyncOpenAI()
judge_llm = llm_factory("gpt-4.1-mini", provider="openai", client=judge_client, max_tokens=4096)
judge_embeddings = RagasOpenAIEmbeddings(client=judge_client, model="text-embedding-3-small")

faithfulness = Faithfulness(llm=judge_llm)
answer_relevancy = AnswerRelevancy(llm=judge_llm, embeddings=judge_embeddings)
context_precision = ContextPrecisionWithReference(llm=judge_llm)
context_recall = ContextRecall(llm=judge_llm)


async def score_dataset(rows: list[dict]) -> list[dict]:
    async def score_one(row: dict) -> dict:
        f, ar, cp, cr = await asyncio.gather(
            faithfulness.ascore(
                user_input=row["user_input"],
                response=row["response"],
                retrieved_contexts=row["retrieved_contexts"],
            ),
            answer_relevancy.ascore(user_input=row["user_input"], response=row["response"]),
            context_precision.ascore(
                user_input=row["user_input"],
                reference=row["reference"],
                retrieved_contexts=row["retrieved_contexts"],
            ),
            context_recall.ascore(
                user_input=row["user_input"],
                retrieved_contexts=row["retrieved_contexts"],
                reference=row["reference"],
            ),
        )
        return {
            "faithfulness": f.value,
            "answer_relevancy": ar.value,
            "context_precision": cp.value,
            "context_recall": cr.value,
        }

    return await asyncio.gather(*(score_one(row) for row in rows))

In [16]:
openai_scores = await score_dataset(openai_rows[:2])

In [17]:
# All 4 metrics x 2 questions run concurrently via asyncio.gather, so each of these is fast.
groq_scores = await score_dataset(groq_rows[:2])

In [ ]:
fireworks_scores = await score_dataset(fireworks_rows[:3])

## 4. Compare quality metrics

In [18]:
quality_cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
# fw_scores = pd.DataFrame(fireworks_scores)[quality_cols].mean()
groq_q_scores = pd.DataFrame(groq_scores)[quality_cols].mean()
oa_scores = pd.DataFrame(openai_scores)[quality_cols].mean()

# comparison = pd.DataFrame({"fireworks": fw_scores, "groq": groq_q_scores, "openai": oa_scores})
comparison = pd.DataFrame({"groq": groq_q_scores, "openai": oa_scores})
comparison

,groq,openai
faithfulness,1.000000,1.000000
answer_relevancy,0.706419,0.737549
context_precision,0.875000,0.875000
context_recall,1.000000,1.000000


## 5. Cost & token comparison via LangSmith

In [19]:
# $ per 1M tokens: (input, output). Fireworks confirmed at docs.fireworks.ai/serverless/pricing;
# Groq confirmed at console.groq.com/docs/models. gpt-4.1-mini / text-embedding-3-small are
# legacy OpenAI models no longer on the live pricing page -- rates below are the
# last-published figures; reconcile against your OpenAI billing dashboard if this drifts.
PRICING = {
    # "fireworks_chat": (0.07, 0.30),       # gpt-oss-20b on Fireworks
    # "fireworks_embedding": (0.10, 0.0),   # qwen3-embedding-8b (input-only)
    "groq_chat": (0.075, 0.30),           # openai/gpt-oss-20b on Groq
    "openai_chat": (0.40, 1.60),          # gpt-4.1-mini
    "openai_embedding": (0.02, 0.0),      # text-embedding-3-small (input-only, shared by groq/openai)
}

In [20]:
from langsmith import Client

client = Client()
project_name = os.environ["LANGSMITH_PROJECT"]


def summarize_provider_runs(tag: str) -> dict:
    prompt_tokens = completion_tokens = 0
    langsmith_cost = 0.0
    run_count = 0
    for run in client.list_runs(project_name=project_name, run_type="llm", filter=f'has(tags, "{tag}")'):
        prompt_tokens += run.prompt_tokens or 0
        completion_tokens += run.completion_tokens or 0
        langsmith_cost += float(run.total_cost or 0)
        run_count += 1
    return {
        "runs": run_count,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "langsmith_reported_cost": langsmith_cost,
    }




openai_usage = summarize_provider_runs("openai")
print("OpenAI:", openai_usage)


groq_usage = summarize_provider_runs("groq")
print("Groq:", groq_usage)

# fireworks_usage = summarize_provider_runs("fireworks")
# print("Fireworks:", fireworks_usage)

OpenAI: {'runs': 42, 'prompt_tokens': 133341, 'completion_tokens': 4817, 'langsmith_reported_cost': 0.05873960000000001}
Groq: {'runs': 8, 'prompt_tokens': 26678, 'completion_tokens': 2861, 'langsmith_reported_cost': 0.0}


In [21]:
# langsmith_reported_cost is 0 unless your LangSmith workspace has pricing configured for
# these exact model IDs (Fireworks/Groq-hosted models usually aren't in its default price
# map), so manual_estimated_cost -- generator/chat tokens only -- is the reliable number.
def manual_cost(usage: dict, chat_key: str) -> float:
    chat_in, chat_out = PRICING[chat_key]
    return (usage["prompt_tokens"] * chat_in + usage["completion_tokens"] * chat_out) / 1_000_000


cost_comparison = pd.DataFrame(
    {
        # "fireworks": {
        #     **fireworks_usage,
        #     "manual_estimated_cost": manual_cost(fireworks_usage, "fireworks_chat"),
        # },
        "groq": {
            **groq_usage,
            "manual_estimated_cost": manual_cost(groq_usage, "groq_chat"),
        },
        "openai": {
            **openai_usage,
            "manual_estimated_cost": manual_cost(openai_usage, "openai_chat"),
        },
    }
)
cost_comparison

,groq,openai
runs,8.000000,42.000000
prompt_tokens,26678.000000,133341.000000
completion_tokens,2861.000000,4817.000000
langsmith_reported_cost,0.000000,0.058740
manual_estimated_cost,0.002859,0.061044
